# Dataset generation & analysis

## Setup

### Imports

In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import itertools

import pyhmmer
import pandas as pd
import numpy as np
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import torch
from tqdm.auto import tqdm
import esm

### HMM profiles

In [2]:
os.makedirs('../data/hmms', exist_ok=True)

!wget -O ../data/hmms/PF00069.hmm.gz "https://www.ebi.ac.uk/interpro/wwwapi/entry/pfam/PF00069?annotation=hmm"
!gunzip -f ../data/hmms/PF00069.hmm.gz
!file ../data/hmms/PF00069.hmm

!wget -O ../data/hmms/PF07714.hmm.gz "https://www.ebi.ac.uk/interpro/wwwapi/entry/pfam/PF07714?annotation=hmm"
!gunzip -f ../data/hmms/PF07714.hmm.gz
!file ../data/hmms/PF07714.hmm

--2026-08-24 13:31:23--  https://www.ebi.ac.uk/interpro/wwwapi/entry/pfam/PF00069?annotation=hmm
Resolving www.ebi.ac.uk (www.ebi.ac.uk)... 193.62.193.80
Connecting to www.ebi.ac.uk (www.ebi.ac.uk)|193.62.193.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 25382 (25K) [application/gzip]
Saving to: ‘../data/hmms/PF00069.hmm.gz’

../data/hmms/PF0006 100%[===================>]  24.79K  --.-KB/s    in 0.1s    

2026-08-24 13:31:25 (189 KB/s) - ‘../data/hmms/PF00069.hmm.gz’ saved [25382/25382]

../data/hmms/PF00069.hmm: ASCII text
--2026-08-24 13:31:26--  https://www.ebi.ac.uk/interpro/wwwapi/entry/pfam/PF07714?annotation=hmm
Resolving www.ebi.ac.uk (www.ebi.ac.uk)... 193.62.193.80
Connecting to www.ebi.ac.uk (www.ebi.ac.uk)|193.62.193.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24895 (24K) [application/gzip]
Saving to: ‘../data/hmms/PF07714.hmm.gz’

../data/hmms/PF0771 100%[===================>]  24.31K  --.-KB/s    in 0.1s    


In [3]:
KINASE_HMM_DIR = Path('../data/hmms')

def read_hmms(hmm_dir: Path) -> list[pyhmmer.plan7.HMM]:
    hmms = []
    for hmm_file in hmm_dir.glob('*.hmm'):
        with pyhmmer.plan7.HMMFile(hmm_file) as f:
            hmms.append(f.read())
    return hmms

kinase_hmms = read_hmms(KINASE_HMM_DIR)
print(f'{len(kinase_hmms)} kinase HMM(s) loaded')

2 kinase HMM(s) loaded


## Generation

### Load data

In [4]:
SEQS_DIR = Path('../data/sequences')

def load_sequences():
    sequences = []

    for fname in SEQS_DIR.glob('*'):
        species_name = fname.stem
        with open(fname) as handle:
            for rec in SeqIO.parse(handle, 'fasta'):
                seq = rec.seq.rstrip('*')
                sequences.append({
                    'species': species_name,
                    'gene': rec.id,
                    'seq': str(seq)
                })

    return sequences

df = pd.DataFrame(load_sequences())
print(f'{len(df)} sequences loaded')

249635 sequences loaded


In [5]:
GENES_LIST_PATH = '../data/list.txt'

with open(GENES_LIST_PATH) as f:
    gene_ids = set(f.read().strip().split('\n'))

print(f'{len(gene_ids)} LRK10L gene IDs loaded')

367 LRK10L gene IDs loaded


In [6]:
df['label'] = pd.to_numeric(df['gene'].isin(gene_ids))

### Length filtering

Kinases are longer than 250aa.

In [7]:
MIN_SEQUENCE_LENGTH = 250

def filter_length():
    old_len = len(df)
    new_df = df[df['seq'].str.len() >= MIN_SEQUENCE_LENGTH]

    print(f'Dropped {old_len - len(new_df)} rows with length <{MIN_SEQUENCE_LENGTH}')

    return new_df

df = filter_length()

Dropped 101983 rows with length <250


### Kinase filtering

In [8]:
ALPHABET = pyhmmer.easel.Alphabet.amino()

def digital_seq_from_str(seq_str, gene_id):
    return pyhmmer.easel.TextSequence(
        name=gene_id,
        sequence=seq_str
    ).digitize(ALPHABET)

def get_kinase_ids():
    seq_block = pyhmmer.easel.DigitalSequenceBlock(ALPHABET)
    for row in df.itertuples(index=False):
        seq_block.append(digital_seq_from_str(row.seq, row.gene))

    hit_ids = set()
    for hmm in kinase_hmms:
        try:
            pipeline = pyhmmer.plan7.Pipeline(hmm.alphabet, bit_cutoffs='gathering')
            hits = pipeline.search_hmm(hmm, seq_block)
        except pyhmmer.errors.MissingCutoffs:
            pipeline = pyhmmer.plan7.Pipeline(hmm.alphabet, E=0.001)
            hits = pipeline.search_hmm(hmm, seq_block)
        except pyhmmer.errors.AlphabetMismatch:
            print(f'{seq_block} is not protein, skipped.')
            continue

        for hit in hits.included:
            hit_ids.add(hit.name)

    return hit_ids

def filter_kinase():
    old_len = len(df)
    new_df = df[df['gene'].isin(get_kinase_ids())]
    print(f'Dropped {old_len - len(new_df)} rows non-kinase')
    return new_df

df = filter_kinase()

Dropped 138707 rows non-kinase


### Check missing known positives

In [9]:
def check_missing_positives():
    missing = gene_ids - set(df['gene'])
    print(f'{len(missing)} LRK10L genes missing from sequences:')
    if missing:
        print(list(missing)[:10])

# TODO Glyma needs to be fixed
check_missing_positives()

98 LRK10L genes missing from sequences:
['Glyma07g10601.1', 'Glyma13g09820.1', 'Glyma09g31421.1', 'Glyma02g33910.2', 'Glyma07g16440.1', 'Glyma07g10490.2', 'Glyma17g32766.1', 'Glyma02g09750.2', 'Glyma20g25471.1', 'Glyma20g25260.2']


### Dedup

Deduplication via `cd-hit` clustering is done only on the severely underrepresented positive data.

Generates and saves a list of genes to drop; original `df` is untouched.

In [10]:
LOAD_DUPLICATE_GENES = True  # Load from disk or rebuild

In [11]:
DUPE_GENES_PATH = '../saves/duplicate_genes.csv'
CDHIT_DIR = Path('../intermediate/cdhit')
CDHIT_C = 0.90

def cdhit(species_name, species_group):
    if len(species_group) < 2:  # cannot cluster single sequence
        return {'species': species_name, 'n_input': len(species_group), 'n_clusters': None, 'redundancy': None}

    os.makedirs(CDHIT_DIR, exist_ok=True)

    fasta_path = CDHIT_DIR / f'{species_name}_positives.fa'
    out_path = CDHIT_DIR / species_name

    SeqIO.write([SeqRecord(Seq(r.seq), id=r.gene, description='') for r in species_group.itertuples(index=False)], fasta_path, 'fasta')

    !cd-hit -i {fasta_path} -o {out_path} -c {CDHIT_C} -n 5 -d 0 -aL 0.8 -aS 0.8 > /dev/null

    n_clusters = sum(1 for _ in SeqIO.parse(out_path, 'fasta'))
    redundancy = 1 - n_clusters / len(species_group)
    return {
        'species': species_name,
        'n_input': len(species_group),
        'n_clusters': n_clusters,
        'redundancy': redundancy,
    }

def clusters_to_dataframe(clstr_path, species_name):
    clusters = []
    current = None
    with open(clstr_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>Cluster'):
                if current is not None:
                    clusters.append(current)
                current = []
            else:
                _, rest = line.split('\t')
                length = int(rest.split(',')[0].strip().rstrip('aa'))
                seq_id = rest.split('>')[1].split('...')[0]
                is_rep = '*' in rest
                identity = None if is_rep else rest.split('at')[-1].strip()
                current.append({
                    'gene': seq_id,
                    'length': length,
                    'is_representative': is_rep,
                    'identity_to_rep': identity,
                })
        if current is not None:
            clusters.append(current)
    rows = []
    for i, cluster in enumerate(clusters):
        for m in cluster:
            rows.append({
                'species': species_name,
                'cluster_id': i,
                'cluster_size': len(cluster),
                **m,
            })
    return pd.DataFrame(rows)

def dedup():
    cdhit_results = []
    for species_name, species_group in df.groupby('species'):
        cdhit_res = cdhit(species_name, species_group[species_group['label'] == 1])
        cdhit_results.append(cdhit_res)

    print('Redundancy:')
    display(pd.DataFrame(cdhit_results).sort_values('redundancy', ascending=False))

    cluster_dfs = []
    for species_name in df['species'].unique():
        clstr_path = CDHIT_DIR / f'{species_name}.clstr'
        if not os.path.exists(clstr_path):
            continue
        cluster_dfs.append(clusters_to_dataframe(clstr_path, species_name))
    cluster_df = pd.concat(cluster_dfs, ignore_index=True)

    return cluster_df.loc[(~cluster_df['is_representative']) & (cluster_df['cluster_size'] > 1), 'gene']

if LOAD_DUPLICATE_GENES:
    duplicate_genes = pd.read_csv(DUPE_GENES_PATH)['gene']
else:
    duplicate_genes = dedup()
    duplicate_genes.to_csv(DUPE_GENES_PATH, index=False, header=['gene'])

print(f'{len(duplicate_genes)} rows are redundant')
print(duplicate_genes.head())

37 rows are redundant
0    VIT_216s0148g00260.1
1    VIT_200s0258g00040.1
2    VIT_216s0050g02720.1
3    VIT_216s0039g01200.1
4    VIT_216s0098g00120.1
Name: gene, dtype: str


### ESM embedding

In [12]:
LOAD_ESM_DF = True  # Load from disk or rebuild

In [13]:
ESM_MODEL_NAME = 'esm2_t30_150M_UR50D'
ESM_DATASET_PATH = '../saves/dataset_esm.csv'

def get_esm_model(device):
    esm_model, esm_alphabet = esm.pretrained.esm2_t30_150M_UR50D()
    esm_model.eval()
    esm_model = esm_model.to(device)
    return esm_model, esm_alphabet.get_batch_converter()

def get_embeddings(esm_model, batch_converter, device, batch_size=8, max_len=1022):
    esm_model.eval()
    embeddings = {}

    # ESM has a length limit; truncate very long sequences
    data = [(r.gene, r.seq[:max_len]) for r in df.itertuples(index=False)]
    data = sorted(data, key=lambda x: len(x[1]))

    for i in tqdm(range(0, len(data), batch_size), desc="Embedding batches"):
        batch = data[i:i+batch_size]
        _, _, tokens = batch_converter(batch)
        tokens = tokens.to(device)

        with torch.no_grad():
            out = esm_model(tokens, repr_layers=[esm_model.num_layers])
            reps = out['representations'][esm_model.num_layers]  # (batch, seq_len, embed_dim)

        for j, (label, seq) in enumerate(batch):
            # mean-pool over real residues, skipping BOS/EOS/padding tokens
            seq_len = len(seq)
            emb = reps[j, 1:seq_len+1].mean(0)  # +1 to skip BOS token
            embeddings[label] = emb.cpu().numpy()

    return embeddings

def esm_dataset():
    device = torch.device(
        'mps' if torch.backends.mps.is_available()
        else 'cuda' if torch.cuda.is_available()
        else 'cpu'
    )
    print('Device:', device)
    
    esm_model, batch_converter = get_esm_model(device)
    embeddings = get_embeddings(esm_model, batch_converter, device)

    emb_dim = next(iter(embeddings.values())).shape[0]
    emb_matrix = np.stack([embeddings[gene] for gene in df['gene']])
    emb_cols = pd.DataFrame(
        emb_matrix,
        columns=[f'esm_{i}' for i in range(emb_dim)],
        index=df.index,
    )

    return pd.concat([df.reset_index(drop=True), emb_cols.reset_index(drop=True)], axis=1)

if LOAD_ESM_DF:
    df_esm = pd.read_csv(ESM_DATASET_PATH)
else:
    df_esm = esm_dataset()
    df_esm.to_csv(ESM_DATASET_PATH, index=False)

print('ESM dataset:', df_esm.shape)
df_esm.head()

ESM dataset: (8945, 644)


,species,gene,seq,label,esm_0,esm_1,esm_2,esm_3,esm_4,esm_5,...,esm_630,esm_631,esm_632,esm_633,esm_634,esm_635,esm_636,esm_637,esm_638,esm_639
0,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_24043.1.p1,WLCITTQTQKLNMKNLERRITLVIIIMFVHFTASLAIIGTDEIALL...,False,-0.108978,-0.073140,-0.096568,-0.203365,0.074926,-0.034012,...,-0.062629,0.111783,-0.145721,0.036615,-0.052408,0.011184,-0.168351,0.034375,-0.175060,0.098445
1,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_25906.1.p1,MLRKTCKMNLEFVCLLLLLSWLKVLDHSIAATSLAKPGCEERCGNL...,False,-0.051743,-0.003080,-0.001061,-0.078826,0.004703,-0.019327,...,-0.056906,0.069938,-0.224253,-0.004619,-0.055436,0.126528,-0.120996,0.078235,-0.106939,0.048921
2,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_26245.1.p1,MLSDLHTTVILFVILCYLDMPISLGQDDEQYRSCGEPFRCGSMDIV...,True,-0.059590,-0.007336,-0.021861,-0.109931,0.001727,-0.035662,...,-0.077579,0.078597,-0.175661,-0.010940,-0.073713,0.105934,-0.174750,0.059239,-0.104596,0.074747
3,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_26268.1.p1,MASGSIMSSAAGKPITDVVLIDNLPKEINAMKINDDKEGKEMEAAV...,False,-0.135528,0.037328,0.073279,-0.121510,-0.060559,-0.052883,...,-0.075612,0.008499,-0.089771,-0.053778,-0.137369,0.010480,-0.045366,0.040035,-0.264432,-0.065106
4,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_26328.1.p1,MFLDSNELKMNYSHLIFVFTIILAYSSVSILAQQPYFGTGTNDCSS...,False,-0.057697,-0.019242,-0.091726,-0.091741,-0.047440,-0.013892,...,-0.063160,0.084078,-0.114984,-0.043391,-0.125841,0.114626,-0.098426,0.043996,-0.099936,0.082743


### K-mer vectors

In [14]:
LOAD_KMER_DF = True  # Load from disk or rebuild

In [15]:
KMER_DATASET_PATH = '../saves/dataset_kmer.csv'
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
K = 2

def kmer_freq_vector(seq, k, kmer_index):
    seq = str(seq).upper()
    vec = np.zeros(len(kmer_index))
    count = 0
    for i in range(len(seq) - k + 1):
        kmer = seq[i:i+k]
        if kmer in kmer_index:  # skip ambiguous
            vec[kmer_index[kmer]] += 1
            count += 1
    if count > 0:
        vec /= count  # normalize
    return vec

def kmer_dataset():
    raw_kmers = [''.join(p) for p in itertools.product(AMINO_ACIDS, repeat=K)]
    kmer_index = {kmer: i for i, kmer in enumerate(raw_kmers)}
    kmer_cols_names = ['kmer_' + k for k in raw_kmers]
    print(f'{len(raw_kmers)} possible {K}-mers')

    kmer_matrix = np.stack([kmer_freq_vector(r.seq, K, kmer_index) for r in df.itertuples(index=False)])
    kmer_cols = pd.DataFrame(
        kmer_matrix,
        columns=kmer_cols_names,
        index=df.index
    )
    return pd.concat([df.reset_index(drop=True), kmer_cols.reset_index(drop=True)], axis=1)

if LOAD_KMER_DF:
    df_kmer = pd.read_csv(KMER_DATASET_PATH)
else:
    df_kmer = kmer_dataset()
    df_kmer.to_csv(KMER_DATASET_PATH, index=False)

print('k-mer dataset:', df_kmer.shape)
df_kmer.head()

k-mer dataset: (8945, 404)


,species,gene,seq,label,kmer_AA,kmer_AC,kmer_AD,kmer_AE,kmer_AF,kmer_AG,...,kmer_YM,kmer_YN,kmer_YP,kmer_YQ,kmer_YR,kmer_YS,kmer_YT,kmer_YV,kmer_YW,kmer_YY
0,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_24043.1.p1,WLCITTQTQKLNMKNLERRITLVIIIMFVHFTASLAIIGTDEIALL...,False,0.001820,0.000910,0.000000,0.001820,0.000910,0.000000,...,0.000910,0.00182,0.000910,0.000000,0.000000,0.001820,0.000000,0.000910,0.0,0.000910
1,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_25906.1.p1,MLRKTCKMNLEFVCLLLLLSWLKVLDHSIAATSLAKPGCEERCGNL...,False,0.007072,0.000000,0.000000,0.002829,0.000000,0.002829,...,0.001414,0.00000,0.002829,0.000000,0.002829,0.002829,0.002829,0.001414,0.0,0.001414
2,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_26245.1.p1,MLSDLHTTVILFVILCYLDMPISLGQDDEQYRSCGEPFRCGSMDIV...,True,0.003049,0.000000,0.000000,0.000000,0.001524,0.003049,...,0.000000,0.00000,0.001524,0.003049,0.001524,0.006098,0.001524,0.003049,0.0,0.000000
3,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_26268.1.p1,MASGSIMSSAAGKPITDVVLIDNLPKEINAMKINDDKEGKEMEAAV...,False,0.004890,0.000000,0.000000,0.004890,0.002445,0.002445,...,0.002445,0.00000,0.000000,0.002445,0.004890,0.004890,0.007335,0.002445,0.0,0.002445
4,Slycopersicum_796_ITAG5.0.protein_primaryTrans...,PRAM_26328.1.p1,MFLDSNELKMNYSHLIFVFTIILAYSSVSILAQQPYFGTGTNDCSS...,False,0.006126,0.001531,0.003063,0.003063,0.003063,0.003063,...,0.001531,0.00000,0.001531,0.001531,0.001531,0.003063,0.001531,0.003063,0.0,0.000000
